# Lesson 02 - Utforska Microsoft Agent Framework

The **Microsoft Agent Framework (MAF)** är ett enhetligt ramverk för att bygga AI-agenter. Det erbjuder en ren, sammansatt arkitektur med fyra kärnbeståndsdelar:

- **Client** – ansluter till en AI-modellendpoint och hanterar kommunikation
- **Agent** – kapslar in en klient med instruktioner och verktygsdefinitioner
- **Tools** – utökar agentens förmågor med anpassade funktioner som modellen kan anropa
- **Session** – underhåller samtalshistorik för flerstegsinteraktioner

I denna lektion bygger vi en **resebokningsagent** som kontrollerar tillgänglighet för destinationer med dessa koncept.


## Installation


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Förstå Agent Framework-arkitekturen

Microsoft Agent Framework följer en lagerarkitektur:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – En `AzureAIProjectAgentProvider` ansluter till en Azure OpenAI-distribution. Den hanterar autentisering, formatering av förfrågningar och tolkning av svar.
2. **Agent** – Skapas från klienten via `provider.create_agent()`, agenten kombinerar modellåtkomst med instruktioner (systemprompt) och verktyg.
3. **Verktyg** – Pythonfunktioner dekorerade med `@tool` som agenten kan anropa för att utföra åtgärder eller hämta data.
4. **Session** – Ett `AgentSession`-objekt (skapat via `agent.create_session()`) som sparar samtalshistorik och möjliggör flerledad dialog där agenten kommer ihåg tidigare kontext.

Låt oss bygga varje lager steg för steg.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Lägga till verktyg med @tool-dekoratören

Verktyg låter agenter utföra åtgärder utöver att generera text. `@tool`-dekoratören omvandlar en vanlig Python-funktion till något som agenten kan anropa.

Viktiga punkter:
- Använd `Annotated[type, "description"]` så att modellen förstår varje parameter.
- Docstringen blir verktygsbeskrivningen som modellen ser.
- `approval_mode="never_require"` betyder att verktyget körs automatiskt utan användarens bekräftelse.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Skapa en agent med verktyg

Nu kombinerar vi klienten, instruktionerna och verktygen till en agent. `instructions` fungerar som systemprompten — de definierar agentens personlighet och beteende.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Fler-turns konversationer med sessioner

En `AgentSession` (skapad via `agent.create_session()`) håller reda på alla meddelanden i en konversation. Genom att skicka samma session till varje `agent.run()`-anrop får agenten tillgång till hela konversationshistoriken och kan hänvisa till tidigare meddelanden.

Vi skickar `tools=[check_destination_availability]` så att agenten kan använda vår tillgänglighetskontroll vid varje tur.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Sammanfattning

I denna lektion utforskade du de fyra pelarna i Microsoft Agent Framework:

| Koncept | Vad du lärde dig |
|---------|------------------|
| **Klient** | `AzureAIProjectAgentProvider` ansluter till Azure OpenAI med autentisering baserad på referenser |
| **Agent** | `provider.create_agent()` paketerar en modellanslutning med instruktioner och ett namn |
| **Verktyg** | Dekoratören `@tool` exponerar Python-funktioner som agenten kan anropa |
| **Session** | `agent.create_session()` bevarar konversationshistorik över flera vändningar |

Dessa byggstenar kombineras för att skapa agenter som kan föra naturliga samtal, anropa externa funktioner och bevara kontext — grunden för mer avancerade agentmönster i senare lektioner.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi eftersträvar noggrannhet, vänligen observera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål ska betraktas som den auktoritativa källan. För viktig information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår vid användning av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
